In [1]:
import os
import gc
import sys
import time
import math
import json
import random
import numpy as np
import pandas as pd
import typing as tp
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

import ast
from pathlib import Path
from itertools import islice
from collections import Counter, defaultdict

import wave
import soundfile

import librosa
import soundfile as sf
from IPython.display import Audio, display

from scipy.sparse import coo_matrix

import shutil

import glob
import warnings
warnings.filterwarnings("ignore")

In [2]:
class CFG():
    SEED          = 42
    N_FOLDS       = 5
    FOLDS_LIST    = [1]
    base_dir      = "/kaggle/input/competitions/birdclef-2026"
    data_dir      = "/kaggle/input/datasets/tatsuyayamamoto/bird-2026-5fold-df-and-ss/data"
    # Librosa
    FS            = 32_000
    N_FFT         = 1_024
    HOP_LEN       = 512
    N_MELS        = 128
    FMIN          = 50
    FMAX          = 14_000
    DURATION      = 5
    # other
    debug         = False
cfg = CFG()
print(f"Debug : {cfg.debug}")


Debug : False


### SEED Everything

In [3]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    print(f"SEED is {seed}")
    
seed_everything(cfg.SEED)

SEED is 42


### Make Directry

In [4]:
DATA = "./data"
if not os.path.exists(DATA):
    os.makedirs(DATA)

AUDIO = "./audio"
if not os.path.exists(AUDIO):
    os.makedirs(AUDIO)

In [5]:
def safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

def load_df(path):
    df = pd.read_csv(path)
    df["labels"] = df["labels"].apply(safe_literal_eval)
    return df

In [6]:
train = load_df(os.path.join(cfg.data_dir, "train.csv"))
ss_df = load_df(os.path.join(cfg.data_dir, "ss.csv"))
print(f"Train Shape: {train.shape}")
print(f"SS    Shape: {ss_df.shape}")

Train Shape: (35379, 12)
SS    Shape: (425, 10)


In [7]:
ss_df.head(2)

,filename,audio_id,labels,class_name,is_ss,weight,duration,n_labels,group_id,class_count
0,BC2026_Train_0001_S08_20250606_030007.ogg,BC2026_Train_0001_S08_20250606_030007_0-5,"[47158son13, 47158son17, 47158son22, 47158son2...",Insecta,1,0.178885,5.0,5,BC2026_Train_0001_S08_20250606_030007,212
1,BC2026_Train_0001_S08_20250606_030007.ogg,BC2026_Train_0001_S08_20250606_030007_20-50,"[47158son10, 47158son13, 47158son17, 47158son2...",Insecta,1,0.377964,30.0,7,BC2026_Train_0001_S08_20250606_030007,212


In [8]:
ss_df["duration"].value_counts().sort_index()

duration
5.0     286
10.0     82
15.0     26
20.0      8
25.0      4
30.0     19
Name: count, dtype: int64

In [9]:
s = ss_df["audio_id"].iloc[0]
s.split("_")[-1].rsplit()

['0-5']

In [10]:
def split_segment_id(audio_id, duration):
    if "_" in audio_id and "-" in audio_id.split("_")[-1]:
        seg = audio_id.split("_")[-1]
        start, end = map(float, seg.split("-"))
    else:
        base = audio_id
        start = 0.0
        end = duration
    return start, end

In [11]:
ss_df[["start", "end"]] = ss_df[["audio_id", "duration"]].apply(
    lambda x: pd.Series(split_segment_id(x[0], x[1])),
    axis=1
)

In [12]:
ss_df[["audio_id", "start", "end"]]

,audio_id,start,end
0,BC2026_Train_0001_S08_20250606_030007_0-5,0.0,5.0
1,BC2026_Train_0001_S08_20250606_030007_20-50,20.0,50.0
2,BC2026_Train_0001_S08_20250606_030007_5-20,5.0,20.0
3,BC2026_Train_0001_S08_20250606_030007_50-55,50.0,55.0
4,BC2026_Train_0001_S08_20250606_030007_55-60,55.0,60.0
...,...,...,...
420,BC2026_Train_0066_S23_20241124_044002_30-35,30.0,35.0
421,BC2026_Train_0066_S23_20241124_044002_35-45,35.0,45.0
422,BC2026_Train_0066_S23_20241124_044002_45-55,45.0,55.0
423,BC2026_Train_0066_S23_20241124_044002_5-20,5.0,20.0


### SS DF Save

In [13]:
ss_df.to_csv(os.path.join(DATA, "ss_df.csv"), index=False)

### Audio Save

In [14]:
unique_df = ss_df[["audio_id", "filename"]].drop_duplicates()
len(unique_df)

425

In [15]:
skip_ids   = []
file_type  = "train_soundscapes"
AUDIO_SS   = os.path.join(AUDIO, f"SS")
os.makedirs(AUDIO_SS, exist_ok=True)

for _, row in tqdm(unique_df.iterrows(), total=len(unique_df), desc="Audio save"):
    filename = row.filename
    audio_id = row.audio_id

    wav, sr = sf.read(os.path.join(cfg.base_dir, file_type, filename), dtype="float32")
    save_path = os.path.join(AUDIO_SS, f"{audio_id}.wav")

    try:
        if wav is None or len(wav) == 0:
            print("EMPTY:", audio_id)
            continue

        if np.isnan(wav).any():
            print("NaN:", audio_id)
            continue

        wav = np.asarray(wav, dtype=np.float32)

        # 👇 subtype変更（超重要）
        sf.write(save_path, wav, samplerate=sr, subtype="PCM_16")
        # FLOATは重い＆不安定
	    # PCM_16は軽くて安定（音声系の標準）

    except Exception as e:
        print("SKIP:", audio_id)
        print("shape:", wav.shape if wav is not None else None)
        print("dtype:", wav.dtype if wav is not None else None)
        print(e)
        skip_ids.append(audio_id)

Audio save:   0%|          | 0/425 [00:00<?, ?it/s]

In [16]:
print(len(unique_df))
print(len(glob.glob(os.path.join(AUDIO_SS, "**/*.wav"), recursive=True)))

425
425


In [17]:
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free: {free/1e9:.2f} GB")

Free: 19.31 GB
